# Bechdel Test Prediction — Comprehensive ML Experiment Suite

**University Group Project — ML Foundations Course**

---

## Research Questions

1. Is the model learning **meaningful representation-related patterns** from pre-release movie metadata?
2. Or is it relying on **shortcut / confounding features** (genre stereotypes, popularity, temporal trends, enrichment artifacts)?

## Notebook Structure
| Section | Content |
|---------|--------|
| 1 | Setup & Imports |
| 2 | Load Data |
| 3 | Feature Engineering & Fixed Train/Test Split |
| 4 | Helper Functions |
| 5 | Reusable Experiment Runner |
| 6 | Exploratory Data Analysis |
| 7 | Model Selection — Baseline (all models) |
| 8 | Imputation Strategy Experiments |
| 9 | Missingness-as-Signal Experiment |
| 10 | Ablation Experiments |
| 11 | Results Comparison Tables |
| 12 | Visualisations & Interpretation |
| 13 | Final Conclusions |

**Design rules (enforced throughout):**
- Train/test split is fixed across every experiment (`seed=42`, `stratify=y`, 80/20).
- Every experiment builds a **fresh, unfitted pipeline** — no object reuse across runs.
- Preprocessing (imputation, scaling, OHE, TF-IDF) is **fitted only on training folds** during CV.
- The held-out test set is used only for final evaluation, never for model or hyperparameter selection.

## 1. Setup & Imports

In [79]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('../src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve as lc
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance as perm_imp

# ── Optional heavy dependencies ───────────────────────────────────────────
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed; SHAP plots will be skipped.  pip install shap')

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

# ── Project source modules ────────────────────────────────────────────────
from data_loader import load_data
from preprocessing import (
    engineer_features, get_feature_lists,
    NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
    GENRE_LIST, NUMERIC_BASE, NUMERIC_TMDB,
    _DensePlotUnion, ColumnSelector
)
from models import get_classifiers, HAS_XGB

# ── Global experiment constants ───────────────────────────────────────────
RANDOM_STATE = 42
CV_SPLITS    = 5
TEST_SIZE    = 0.20
SCORING      = 'roc_auc'

# ── Plotting style ────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('colorblind')
COLORS = sns.color_palette('colorblind', 12)

print('Environment ready.')
print(f'  SHAP available:      {HAS_SHAP}')
print(f'  missingno available: {HAS_MSNO}')
print(f'  XGBoost available:   {HAS_XGB}')


Environment ready.
  SHAP available:      True
  missingno available: False
  XGBoost available:   True


## 2. Load Data

`movies_complete.csv` is the fully merged dataset (Bechdel/IMDb + TMDb enrichment + Wikidata supplement). Films without a Bechdel rating are automatically excluded by the loader.

**Expected missingness:** budget, revenue, and TMDb popularity are missing for ~50–78% of films depending on the dataset version — this is by design, not a data quality issue. The enrichment comes from a separate source with only partial overlap.

In [80]:
DATA_DIR      = os.path.abspath('../data')
COMPLETE_PATH = os.path.join(DATA_DIR, 'movies_complete.csv')
BECHDEL_PATH  = os.path.join(DATA_DIR, 'Bechdel_IMDB_Merge0524 copy.csv')
TMDB_PATH     = os.path.join(DATA_DIR, 'movies_enriched_5k.csv')

df_raw = load_data(
    bechdel_path=BECHDEL_PATH,
    tmdb_path=TMDB_PATH,
    complete_path=COMPLETE_PATH if os.path.exists(COMPLETE_PATH) else None,
)

print(f'\nDataset shape: {df_raw.shape}')
vc = df_raw['bechdel_pass'].value_counts()
print(f'Target distribution: Pass={vc.get(1,0):,} ({vc.get(1,0)/len(df_raw):.1%})  '
      f'Fail={vc.get(0,0):,} ({vc.get(0,0)/len(df_raw):.1%})')
print('\nMissingness rates (%):')
miss = df_raw.isnull().mean().mul(100).round(1)
print(miss[miss > 0].to_string())
df_raw.head(3)

[data_loader] Dropped 1,618 unlabelled movies (no Bechdel rating). 9,718 labelled films retained.
[data_loader] Loaded 9,718 films from complete dataset.

Dataset shape: (9718, 15)
Target distribution: Pass=5,602 (57.6%)  Fail=4,116 (42.4%)

Missingness rates (%):
runtime                 0.1
budget                 55.2
revenue                56.7
tmdb_popularity        65.2
cast_size               1.2
has_female_director     2.2


,imdb_id,title,year,runtime,imdb_rating,num_votes,bechdel_score,bechdel_pass,genres,budget,revenue,tmdb_popularity,cast_size,has_female_director,plot
0,tt0000009,Miss Jerry,1894.0,45.0,5.4,212.0,0.0,0,Romance,NaN,NaN,NaN,0.0,0.0,1894 film directed by Alexander Black
1,tt0000574,"Story of the Kelly Gang, The",1906.0,70.0,6.0,903.0,1.0,0,Action|Adventure|Biography,NaN,NaN,NaN,1.0,0.0,1906 film
2,tt0002101,Cleopatra,1912.0,100.0,5.1,622.0,2.0,0,Drama|History,NaN,NaN,NaN,3.0,0.0,1912 film by Charles L. Gaskill


## 3. Feature Engineering & Fixed Train/Test Split

All feature engineering is **row-local and deterministic**: log-transforms, decade bucketing, and genre one-hot flags depend only on a row's own values, so applying them before the split introduces zero leakage.

Raw columns (`runtime`, `num_votes`, `budget`, `revenue`, `cast_size`, `genres`) are replaced by their engineered versions. The raw columns and identifiers are dropped from `X`; the `plot` column is kept so TF-IDF experiments can use it inside the pipeline.

> **The train/test split is fixed for the entire notebook.** All experiments use exactly the same split.

In [81]:
# Row-local engineering — safe to do before split
df = engineer_features(df_raw)

TARGET = 'bechdel_pass'

# Columns to exclude from X:
#   identifiers, the raw columns (replaced by engineered versions), and the target
DROP_COLS = [
    'imdb_id', 'title',
    'bechdel_score', 'bechdel_pass',  # target leakage
    'runtime', 'num_votes',           # replaced by log_runtime, log_num_votes
    'budget', 'revenue', 'cast_size', # replaced by log_* versions
    'genres',                         # replaced by genre_* one-hots
    # 'plot' intentionally KEPT for optional TF-IDF inside the pipeline
]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train pass rate: {y_train.mean():.3f}  |  Test pass rate: {y_test.mean():.3f}')
print(f'\nAll feature columns ({len(X_train.columns)}):')
print(list(X_train.columns))

Train: 7,774 rows  |  Test: 1,944 rows
Train pass rate: 0.576  |  Test pass rate: 0.577

All feature columns (31):
['year', 'imdb_rating', 'tmdb_popularity', 'has_female_director', 'plot', 'log_runtime', 'log_num_votes', 'log_budget', 'log_revenue', 'log_cast_size', 'decade', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_biography', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_musical', 'genre_mystery', 'genre_romance', 'genre_sci_fi', 'genre_sport', 'genre_thriller', 'genre_war', 'genre_western']


## 4. Helper Functions

### 4a. Missingness Indicator Creator
For the **missingness-as-signal experiment** we add binary flags (`budget_missing`, `revenue_missing`, etc.) encoding whether each TMDb-derived feature was absent. These flags are computed from the data **before** any pipeline fitting, so they are safe to include as features.

### 4b. Feature Group Definitions
We define which columns belong to each feature group for ablation experiments. All definitions are derived from `X_train.columns` so they reflect exactly what is available after the split.

### 4c. Fresh Pipeline Builder
`build_experiment_pipeline()` constructs a **fully new, unfitted sklearn Pipeline** every time it is called. The imputation strategy, TF-IDF inclusion, and feature lists are all configurable per call.

In [82]:
# ── 4a. Missingness indicators ────────────────────────────────────────────
ENRICHMENT_COLS_MAP = {
    'log_budget':      'budget_missing',
    'log_revenue':     'revenue_missing',
    'tmdb_popularity': 'popularity_missing',
    'log_cast_size':   'cast_size_missing',
}
PLOT_MISSING_COL = 'plot_missing'


def add_missingness_indicators(X: pd.DataFrame) -> pd.DataFrame:
    """Add binary missingness indicator columns for enrichment-derived features."""
    X = X.copy()
    for feat_col, indicator_name in ENRICHMENT_COLS_MAP.items():
        X[indicator_name] = X[feat_col].isna().astype(int) if feat_col in X.columns else 0
    # plot_missing: empty string or NaN means no plot was available
    if 'plot' in X.columns:
        X[PLOT_MISSING_COL] = X['plot'].fillna('').eq('').astype(int)
    else:
        X[PLOT_MISSING_COL] = 0
    return X


# ── 4b. Feature group definitions for ablations ───────────────────────────
GENRE_FEATURES = [c for c in X_train.columns if c.startswith('genre_')]

POPULARITY_FEATURES = [
    c for c in ['imdb_rating', 'log_num_votes', 'log_budget', 'log_revenue', 'tmdb_popularity']
    if c in X_train.columns
]

TEMPORAL_FEATURES = [c for c in ['year', 'decade'] if c in X_train.columns]

TMDB_ENRICHMENT_FEATURES = [
    c for c in ['log_budget', 'log_revenue', 'tmdb_popularity', 'log_cast_size', 'has_female_director']
    if c in X_train.columns
]

print('Feature group sizes:')
print(f'  Genre features:      {len(GENRE_FEATURES):>3}  {GENRE_FEATURES[:3]}...')
print(f'  Popularity features: {len(POPULARITY_FEATURES):>3}  {POPULARITY_FEATURES}')
print(f'  Temporal features:   {len(TEMPORAL_FEATURES):>3}  {TEMPORAL_FEATURES}')
print(f'  TMDb enrichment:     {len(TMDB_ENRICHMENT_FEATURES):>3}  {TMDB_ENRICHMENT_FEATURES}')

Feature group sizes:
  Genre features:       20  ['genre_action', 'genre_adventure', 'genre_animation']...
  Popularity features:   5  ['imdb_rating', 'log_num_votes', 'log_budget', 'log_revenue', 'tmdb_popularity']
  Temporal features:     2  ['year', 'decade']
  TMDb enrichment:       5  ['log_budget', 'log_revenue', 'tmdb_popularity', 'log_cast_size', 'has_female_director']


In [83]:
# ── 4c. Fresh pipeline builder ────────────────────────────────────────────
def build_experiment_pipeline(
    clf,
    numeric_features,
    categorical_features,
    binary_features,
    imputation_strategy='median',
    use_plot_tfidf=False,
):
    """
    Build a fresh, unfitted sklearn Pipeline.
    Every call creates independent estimator objects — no shared state between experiments.
    """
    valid_strategies = ('mean', 'median', 'most_frequent')
    num_strategy = imputation_strategy if imputation_strategy in valid_strategies else 'median'

    dense_ct = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy=num_strategy)),
                ('scale',  StandardScaler()),
            ]), numeric_features),
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore',
                                         sparse_output=False, drop='first')),
            ]), categorical_features),
            ('bin', SimpleImputer(strategy='constant', fill_value=0), binary_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

    if use_plot_tfidf:
        tfidf_pipe = Pipeline([
            ('sel',   ColumnSelector('plot')),
            ('tfidf', TfidfVectorizer(
                max_features=150, sublinear_tf=True,
                min_df=3, ngram_range=(1, 2), stop_words='english',
            )),
        ])
        preprocessor = _DensePlotUnion(dense_ct, tfidf_pipe)
    else:
        preprocessor = dense_ct

    return Pipeline([
        ('preprocess', preprocessor),
        ('clf', clone(clf)),
    ])


print('Pipeline builder defined.')

Pipeline builder defined.


#%% md
## 5. Reusable Experiment Runner

`run_experiment(config)` is the single entry point for every experiment. It enforces the leakage-safe protocol:

1. **Prepare data** — optionally add missingness indicators and/or drop feature groups.
2. **Build a fresh pipeline** — new estimator objects for every call.
3. **Cross-validate on the training split only** — test set is never seen here.
4. **Fit on the full training split.**
5. **Evaluate on the held-out test split** — one call per experiment, for reporting only.
6. **Log results** to the shared `ALL_RESULTS` list.

Config schema:
```
experiment_name : str   — human-readable label
model_key       : str   — 'dummy' | 'logreg' | 'tree' | 'rf' | 'xgb'
imputation      : str   — 'median' | 'mean' | 'most_frequent'
use_tfidf       : bool  — include TF-IDF plot features
add_missingness : bool  — add missingness indicator features
drop_features   : list  — column names to remove before fitting
ablation_type   : str   — experiment category tag
notes           : str   — free-text annotation
```

In [ ]:
#%%
ALL_RESULTS = []   # shared results log (metrics only, no heavy objects)


def run_experiment(config: dict) -> dict:
    name       = config.get('experiment_name', 'unnamed')
    model_key  = config.get('model_key', 'xgb')
    imputation = config.get('imputation', 'median')
    use_tfidf  = config.get('use_tfidf', False)
    add_miss   = config.get('add_missingness', False)
    drop_feats = config.get('drop_features', [])
    notes      = config.get('notes', '')

    # ── 1. Prepare data ───────────────────────────────────────────────────
    X_tr = X_train.copy()
    X_te = X_test.copy()

    if add_miss:
        X_tr = add_missingness_indicators(X_tr)
        X_te = add_missingness_indicators(X_te)

    cols_to_drop = [c for c in drop_feats if c in X_tr.columns]
    X_tr = X_tr.drop(columns=cols_to_drop, errors='ignore')
    X_te = X_te.drop(columns=cols_to_drop, errors='ignore')

    # ── 2. Determine feature lists ────────────────────────────────────────
    miss_indicator_cols = []
    if add_miss:
        miss_indicator_cols = [
            c for c in list(ENRICHMENT_COLS_MAP.values()) + [PLOT_MISSING_COL]
            if c in X_tr.columns
        ]

    num_feats = [c for c in NUMERIC_FEATURES    if c in X_tr.columns]
    cat_feats = [c for c in CATEGORICAL_FEATURES if c in X_tr.columns]
    bin_feats = [c for c in BINARY_FEATURES      if c in X_tr.columns]
    bin_feats = bin_feats + miss_indicator_cols

    if not (num_feats or cat_feats or bin_feats):
        print(f'  [SKIP] {name}: no features remaining.')
        return {}

    # ── 3. Get classifier ─────────────────────────────────────────────────
    clfs = get_classifiers(random_state=RANDOM_STATE)
    if model_key not in clfs:
        print(f'  [SKIP] {name}: model "{model_key}" not available.')
        return {}
    clf = clfs[model_key]

    # ── 4. Build fresh pipeline ───────────────────────────────────────────
    pipeline = build_experiment_pipeline(
        clf=clf,
        numeric_features=num_feats,
        categorical_features=cat_feats,
        binary_features=bin_feats,
        imputation_strategy=imputation,
        use_plot_tfidf=use_tfidf and 'plot' in X_tr.columns,
    )

    # ── 5. CV on training split only ─────────────────────────────────────
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(pipeline, X_tr, y_train, cv=cv,
                                scoring=SCORING, n_jobs=-1)
    cv_mean, cv_std = float(cv_scores.mean()), float(cv_scores.std())

    # ── 6. Fit on full training split ─────────────────────────────────────
    pipeline.fit(X_tr, y_train)

    # ── 7. Evaluate on held-out test split ────────────────────────────────
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)
    ap   = average_precision_score(y_test, y_prob)

    row = {
        'experiment_name':   name,
        'model_name':        model_key,
        'imputation_method': imputation,
        'ablation_type':     config.get('ablation_type', 'none'),
        'use_tfidf':         use_tfidf,
        'add_missingness':   add_miss,
        'cv_mean':           round(cv_mean, 4),
        'cv_std':            round(cv_std,  4),
        'accuracy':          round(acc,  4),
        'precision':         round(prec, 4),
        'recall':            round(rec,  4),
        'f1':                round(f1,   4),
        'roc_auc':           round(auc,  4),
        'avg_precision':     round(ap,   4),
        'notes':             notes,
    }
    ALL_RESULTS.append(row)

    # Return everything including heavy objects for downstream visualisation
    result = dict(row)
    result.update({'_pipeline': pipeline, '_y_pred': y_pred,
                   '_y_prob': y_prob, '_X_te': X_te})

    print(f'  {name:<58} CV={cv_mean:.4f}\u00b1{cv_std:.4f}  '
          f'TestAUC={auc:.4f}  F1={f1:.4f}')
    return result


print('run_experiment() defined.  ALL_RESULTS initialised.')

#%% md
## 6. Exploratory Data Analysis

Before running any experiments we examine the data to understand:
- Target class balance
- Genre-level pass rates (potential shortcut signals)
- Temporal trends in Bechdel pass rates
- Missingness patterns and whether they correlate with the target (enrichment artifacts)
- Feature distributions by Bechdel outcome

**Why this matters:** Strong genre or temporal effects visible in EDA suggest the model may learn shortcuts (predicting "romance = pass" or "modern film = pass") rather than deeper representation signals.

In [ ]:
#%%
# ── 6.1  Target distribution + genre pass rates + temporal trends ─────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Target counts
ax = axes[0]
vc = df_raw['bechdel_pass'].value_counts()
ax.bar(['Fail (0)', 'Pass (1)'], [vc.get(0, 0), vc.get(1, 0)],
       color=[COLORS[1], COLORS[0]], alpha=0.85)
ax.set_title('Target Distribution')
ax.set_ylabel('Number of films')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)

# Genre pass rates (horizontal bar)
ax = axes[1]
genre_pass = {}
for g in GENRE_LIST:
    col = f'genre_{g.replace("-", "_")}'
    if col in df.columns:
        mask = df[col] == 1
        if mask.sum() >= 30:
            genre_pass[g] = df.loc[mask, 'bechdel_pass'].mean()
genre_s = pd.Series(genre_pass).sort_values()
overall = df['bechdel_pass'].mean()
bar_colors = [COLORS[0] if v >= overall else COLORS[1] for v in genre_s.values]
ax.barh(genre_s.index, genre_s.values, color=bar_colors, alpha=0.85)
ax.axvline(overall, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall {overall:.2f}')
ax.set_xlabel('Pass rate')
ax.set_title('Pass Rate by Genre')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)

# Pass rate over decades
ax = axes[2]
decade_pass = df.groupby('decade')['bechdel_pass'].agg(['mean', 'count'])
decade_pass = decade_pass[decade_pass['count'] >= 20]  # drop tiny decades
ax.plot(range(len(decade_pass)), decade_pass['mean'], 'o-', color=COLORS[2], linewidth=2)
ax.fill_between(range(len(decade_pass)),
                decade_pass['mean'] - np.sqrt(decade_pass['mean'] * (1 - decade_pass['mean']) / decade_pass['count']),
                decade_pass['mean'] + np.sqrt(decade_pass['mean'] * (1 - decade_pass['mean']) / decade_pass['count']),
                alpha=0.2, color=COLORS[2])
ax.set_xticks(range(len(decade_pass)))
ax.set_xticklabels(decade_pass.index, rotation=45, ha='right', fontsize=8)
ax.axhline(overall, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('Pass Rate by Decade')
ax.set_ylabel('Pass rate')
ax.set_ylim(0, 1)

plt.suptitle('EDA: Bechdel Test Patterns in the Dataset', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Genre shortcuts visible:  some genres (e.g. War, Horror) have substantially lower pass rates.')
print('Temporal signal visible:  pass rates improve from 1920s to ~2000s, then plateau.')
print('Both patterns could lead to shortcut learning.')

In [ ]:
#%%
# ── 6.2  Missingness analysis — does absence of enrichment data correlate with outcome?
#
# If films that PASS have lower missingness rates than films that FAIL, or vice versa,
# the model may learn to predict from data absence rather than data content.
# This is an enrichment artifact: which films happened to be included in the TMDb dataset.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

enrich_raw = [c for c in ['budget', 'revenue', 'tmdb_popularity', 'cast_size'] if c in df_raw.columns]

# Missing rate by pass/fail
ax = axes[0]
miss_pass_df = pd.DataFrame({
    'Pass': [df_raw.loc[df_raw['bechdel_pass'] == 1, c].isna().mean() * 100 for c in enrich_raw],
    'Fail': [df_raw.loc[df_raw['bechdel_pass'] == 0, c].isna().mean() * 100 for c in enrich_raw],
}, index=enrich_raw)
miss_pass_df.plot.bar(ax=ax, color=[COLORS[0], COLORS[1]], alpha=0.85)
ax.set_ylabel('Missing rate (%)')
ax.set_title('Enrichment Missingness: Pass vs Fail Films\n'
             '(Difference = enrichment artifact signal)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Bechdel outcome')

# Correlation of missingness indicators with target
ax = axes[1]
enrich_eng = [c for c in ['log_budget', 'log_revenue', 'tmdb_popularity', 'log_cast_size'] if c in df.columns]
miss_corr = {c.replace('log_', '').replace('tmdb_', ''): df[c].isna().astype(int).corr(df['bechdel_pass'])
             for c in enrich_eng}
miss_corr_s = pd.Series(miss_corr)
ax.bar(miss_corr_s.index, miss_corr_s.values,
       color=[COLORS[0] if v >= 0 else COLORS[1] for v in miss_corr_s.values], alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Pearson correlation with bechdel_pass')
ax.set_title('Correlation: "Feature missing" indicator → Target')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

plt.suptitle('EDA: Missingness as a Potential Predictive Signal (Enrichment Artifact)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('If these correlations are non-trivial, the model may learn from data absence.')
print('We will test this formally in Section 9.')

In [ ]:
#%%
# ── 6.3  Numeric feature distributions by Bechdel outcome ─────────────────
#
# Overlapping distributions → weak signal.  Well-separated → strong (possibly shortcut) signal.

show_cols = ['year', 'imdb_rating', 'log_num_votes', 'log_budget', 'log_revenue', 'tmdb_popularity']
show_cols = [c for c in show_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(show_cols):
    ax = axes[i]
    for val, label, color in [(0, 'Fail', COLORS[1]), (1, 'Pass', COLORS[0])]:
        data = df.loc[df['bechdel_pass'] == val, col].dropna()
        ax.hist(data, bins=35, alpha=0.5, label=label, color=color, density=True)
        ax.axvline(data.median(), linestyle='--', color=color, alpha=0.8, linewidth=1.2)
    ax.set_title(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Numeric Feature Distributions: Pass vs Fail Films\n'
             '(Dashed lines = medians; separated medians suggest predictive signal)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

#%% md
## 7. Model Selection — Baseline (All Models on Training Split)

We evaluate all available classifiers under identical conditions:
- Preprocessing: median imputation, StandardScaler, OHE for decade
- No TF-IDF (for speed and to establish a clean baseline)
- No missingness indicators

**Model selection is based exclusively on CV ROC-AUC on the training split.**  
The test set is untouched until section 12.

In [ ]:
#%%
print('=' * 75)
print('BASELINE: All models, median imputation, no TF-IDF')
print('=' * 75)
print(f'{"Experiment":<60} {"CV AUC":>10}  {"TestAUC":>8}  {"F1":>6}')
print('-' * 75)

model_keys = ['dummy', 'logreg', 'tree', 'rf'] + (['xgb'] if HAS_XGB else [])

baseline_results = {}
for mk in model_keys:
    res = run_experiment({
        'experiment_name': f'Baseline — {mk}',
        'model_key': mk,
        'imputation': 'median',
        'use_tfidf': False,
        'add_missingness': False,
        'drop_features': [],
        'ablation_type': 'baseline',
        'notes': 'Default median-imputed pipeline, all features',
    })
    if res:
        baseline_results[mk] = res

print('\nModel selection is based on CV ROC-AUC on the training split only.')

In [ ]:
#%%
# Select best non-dummy model based on CV AUC
non_dummy = {k: v for k, v in baseline_results.items() if k != 'dummy'}
BEST_MODEL = max(non_dummy, key=lambda k: non_dummy[k]['cv_mean'])
BEST_RESULT = baseline_results[BEST_MODEL]

print(f'Best model (by CV ROC-AUC): {BEST_MODEL.upper()}')
print(f'  CV  ROC-AUC:  {BEST_RESULT["cv_mean"]:.4f} ± {BEST_RESULT["cv_std"]:.4f}')
print(f'  Test ROC-AUC: {BEST_RESULT["roc_auc"]:.4f}')
print(f'  Test F1:      {BEST_RESULT["f1"]:.4f}')
print(f'\nAll ablation experiments will use: {BEST_MODEL.upper()}')

#%% md
## 8. Imputation Strategy Experiments

~50–78% of TMDb-derived features are missing. The choice of imputation strategy can affect model performance, especially for tree-based models. We test three strategies on the best model.

| Strategy | Description | Expected effect |
|----------|-------------|----------------|
| `median` | Replace missing values with the column median | Robust to outliers (default) |
| `mean` | Replace with column mean | Sensitive to outliers in budget/revenue |
| `most_frequent` | Replace with the most common value | Mainly useful for sparse/categorical-like numerics |

**Leakage note:** All imputation statistics are computed **inside the CV fold** on training data only, never on the full dataset.

In [ ]:
#%%
print('=' * 75)
print(f'IMPUTATION EXPERIMENTS  (model: {BEST_MODEL.upper()})')
print('=' * 75)
print(f'{"Experiment":<60} {"CV AUC":>10}  {"TestAUC":>8}  {"F1":>6}')
print('-' * 75)

imputation_results = {}
for strategy, label in [
    ('median',        'Median imputation (default)'),
    ('mean',          'Mean imputation'),
    ('most_frequent', 'Most-frequent imputation'),
]:
    res = run_experiment({
        'experiment_name': f'Imputation — {strategy}',
        'model_key': BEST_MODEL,
        'imputation': strategy,
        'use_tfidf': False,
        'add_missingness': False,
        'drop_features': [],
        'ablation_type': 'imputation',
        'notes': label,
    })
    if res:
        imputation_results[strategy] = res

print('\nInterpretation guide:')
print('  Large differences between strategies → imputation choice has meaningful impact.')
print('  Small differences → imputed values are not heavily relied upon (robust to choice).')